# VoxIndica — SadTalker T4 spike

> **Status: unverified.** This notebook is generated from a template in
> `workers/build_video_notebooks.py` and has not been executed on a T4 by this
> repository. Installation commands and flags are a starting point, not a
> guarantee. If a step fails, keep the real error and report it — do not paper
> over it with a placeholder clip.
>
> **Consent:** use only a presenter photo and voice clip that the person has
> explicitly agreed to use for an AI-generated likeness and synthetic voice.
> Never upload someone else's face.

Goal: one consented photo + one 3–30s WAV → one MP4, with wall time,
peak GPU memory and the real failure mode recorded.

Engine: [OpenTalker/SadTalker](https://github.com/OpenTalker/SadTalker).
This notebook clones the default branch; the commit is printed so the
result can be tied to an exact revision.

## 1. GPU check

In [ ]:
import platform
import shutil
import subprocess
import sys

import torch

print("Python:", sys.version.split()[0], "|", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA device. In Kaggle, open Settings -> Accelerator -> GPU T4 x2, "
        "then restart the kernel."
    )
props = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(props.total_memory / 1024**3, 1), "GB")
print("ffmpeg:", shutil.which("ffmpeg") or "missing")
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
    capture_output=True, text=True
).stdout)

## Inputs

Attach a **private** Kaggle dataset containing:

- one clear, front-facing photo (`jpg`/`png`, at least 512×512, eyes visible)
- one narration clip (`wav`, mono/stereo, 3–30 seconds)

The next cell lists `/kaggle/input`. If automatic discovery picks the wrong
file, set `IMAGE_OVERRIDE` and `AUDIO_OVERRIDE` to absolute paths.

In [ ]:
from pathlib import Path

import soundfile as sf
from IPython.display import Audio, display

IMAGE_OVERRIDE = ""
AUDIO_OVERRIDE = ""

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp"}
AUDIO_SUFFIXES = {".wav"}


def _find(root: Path, suffixes: set[str]) -> list[Path]:
    if not root.is_dir():
        return []
    return sorted(
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in suffixes
    )


def locate_image() -> Path:
    if IMAGE_OVERRIDE:
        path = Path(IMAGE_OVERRIDE)
        if not path.is_file():
            raise FileNotFoundError(f"IMAGE_OVERRIDE not found: {path}")
        return path
    hits = _find(Path("/kaggle/input"), IMAGE_SUFFIXES)
    if not hits:
        raise FileNotFoundError(
            "No image under /kaggle/input. Attach a dataset with a presenter photo "
            "and set IMAGE_OVERRIDE if needed."
        )
    return hits[0]


def locate_audio() -> Path:
    if AUDIO_OVERRIDE:
        path = Path(AUDIO_OVERRIDE)
        if not path.is_file():
            raise FileNotFoundError(f"AUDIO_OVERRIDE not found: {path}")
        return path
    hits = _find(Path("/kaggle/input"), AUDIO_SUFFIXES)
    if not hits:
        raise FileNotFoundError(
            "No .wav under /kaggle/input. Attach a 3-30s narration clip and set "
            "AUDIO_OVERRIDE if needed."
        )
    return hits[0]


IMAGE_PATH = locate_image()
AUDIO_PATH = locate_audio()

data, rate = sf.read(str(AUDIO_PATH))
duration = len(data) / rate
print("Image:", IMAGE_PATH, f"({IMAGE_PATH.stat().st_size} bytes)")
print("Audio:", AUDIO_PATH, f"({duration:.2f}s at {rate} Hz)")
if not 3 <= duration <= 30:
    raise ValueError(
        f"The reference clip is {duration:.1f}s. This spike expects 3-30s; trim it first."
    )
display(Audio(data, rate=rate))

## 2. Install SadTalker
SadTalker's requirements pin several older packages. Pip is allowed to
resolve against Kaggle's image; if it fails, stop and report the pip error.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/kaggle/working/SadTalker")
if not (REPO / "inference.py").is_file():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/OpenTalker/SadTalker.git", str(REPO)],
        check=True,
    )
print("SadTalker commit:")
print(subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    capture_output=True, text=True,
).stdout.strip())
print(subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%cs %s"],
    capture_output=True, text=True,
).stdout.strip())

install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir",
     "-r", str(REPO / "requirements.txt")],
    text=True, capture_output=True,
)
print(install.stdout[-12000:])
print(install.stderr[-6000:])
print("pip exit:", install.returncode)
if install.returncode != 0:
    raise RuntimeError(
        "SadTalker requirements did not install. Report the pip error above; "
        "do not skip the install and pretend the run worked."
    )

### torchvision compatibility
`basicsr` (a SadTalker dependency) imports a module that newer
torchvision releases removed. This cell applies the standard one-line
patch only if the old import is present.

In [ ]:
import importlib.util
from pathlib import Path

# SadTalker pulls basicsr, whose old import of
# torchvision.transforms.functional_tensor breaks on newer torchvision. Apply
# the standard one-line compatibility patch only when that import is present.
spec = importlib.util.find_spec("basicsr")
if spec and spec.origin:
    degradations = Path(spec.origin).parent / "data" / "degradations.py"
    if degradations.is_file():
        text = degradations.read_text(encoding="utf-8")
        old = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
        new = "from torchvision.transforms.functional import rgb_to_grayscale"
        if old in text:
            degradations.write_text(text.replace(old, new), encoding="utf-8")
            print("Patched basicsr:", degradations)
        else:
            print("basicsr needs no functional_tensor patch.")
    else:
        print("basicsr installed but degradations.py was not found; continuing.")
else:
    print("basicsr is not importable; the inference cell will show the real error.")

## 3. Download checkpoints

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/kaggle/working/SadTalker")
script = REPO / "scripts" / "download_models.sh"
if not script.is_file():
    raise FileNotFoundError(f"Missing SadTalker weight script: {script}")

os.chdir(REPO)
result = subprocess.run(["bash", str(script)], text=True, capture_output=True)
print(result.stdout[-12000:])
print(result.stderr[-6000:])
print("weights exit:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("SadTalker checkpoint download failed; see the output above.")

for path in sorted((REPO / "checkpoints").glob("*")):
    print(path.name, path.stat().st_size if path.is_file() else "<dir>")

## Run and measure

The next cell runs the engine once and samples `nvidia-smi` in the background.
`peak_vram_mb` is the highest **total** GPU memory observed, so it includes
CUDA context and any other process on the device — treat it as an upper bound,
not an isolated model footprint.

In [ ]:
import os
import subprocess
import sys
import threading
import time
from pathlib import Path

REPO = Path("/kaggle/working/SadTalker")
OUT = Path("/kaggle/working/sadtalker_out")
OUT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "inference.py",
    "--driven_audio", str(AUDIO_PATH),
    "--source_image", str(IMAGE_PATH),
    "--result_dir", str(OUT),
    "--still",
    "--preprocess", "full",
    "--size", "256",
    "--batch_size", "2",
]

peak = {"mb": 0}
stop = threading.Event()


def sample_vram() -> None:
    while not stop.is_set():
        try:
            raw = subprocess.run(
                ["nvidia-smi", "--query-gpu=memory.used",
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True, timeout=5,
            ).stdout
            values = [int(part) for part in raw.replace(",", " ").split() if part.isdigit()]
            if values:
                peak["mb"] = max(peak["mb"], max(values))
        except Exception:
            pass
        time.sleep(0.5)


os.chdir(REPO)
sampler = threading.Thread(target=sample_vram, daemon=True)
sampler.start()
started = time.time()
proc = subprocess.run(cmd, text=True, capture_output=True)
elapsed = time.time() - started
stop.set()
sampler.join(timeout=3)

print(proc.stdout[-12000:])
print(proc.stderr[-12000:])
print(f"exit={proc.returncode} wall_s={elapsed:.1f} peak_vram_mb={peak['mb']}")
if proc.returncode != 0:
    raise RuntimeError(
        "SadTalker inference failed. The log above is the real error; do not "
        "fabricate a result."
    )

## 4. Inspect the clip

In [ ]:
import shutil
import subprocess
from pathlib import Path

from IPython.display import Video, display

WORK = Path("/kaggle/working")
output_dirs = [WORK / "sadtalker_out", WORK / "musetalk_out"]
clips = sorted(
    (
        p for root in output_dirs if root.is_dir()
        for p in root.rglob("*.mp4")
        if p.stat().st_size > 0
    ),
    key=lambda p: p.stat().st_mtime,
)
if not clips:
    raise FileNotFoundError(
        "No MP4 was produced under the engine output directory. The inference "
        "log above is the real failure; do not continue as if this spike passed."
    )

latest = clips[-1]
final = WORK / "avatar_spike.mp4"
shutil.copy2(latest, final)
probe = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries",
     "format=duration,size:stream=width,height,r_frame_rate",
     "-of", "json", str(final)],
    capture_output=True, text=True,
)
print("Clip:", final)
print("Bytes:", final.stat().st_size)
print(probe.stdout)
display(Video(str(final), embed=False, width=480))

## Report back

Copy these into the spike result table:

| Field | Value |
| --- | --- |
| Engine + version/commit | |
| Wall time (s) | |
| Peak GPU memory (MB, upper bound) | |
| Output duration / resolution | |
| Subjective lip-sync (1–5) | |
| Subjective identity retention (1–5) | |
| Artifacts or failure mode | |

The studio will only integrate an engine after a real clip is produced and these
numbers are recorded. A notebook that errors is a useful result; a fabricated
pass is not.